# Task 1: Data Handling & Memory Management

TIM Milan mobile network traffic (Nov 2013–Jan 2014). This notebook loads the ~5 GB raw dataset efficiently, keeps only **Square id**, **Time Interval**, and **Internet traffic activity**, and reports memory use **before and after** optimization.

**Source:** Harvard Dataverse — Telecommunications activity, Milan [1]. Fields follow [1]: country code is the 3rd column; internet activity is the last column when present.

In [ ]:
# Standard libraries for paths, memory, and system info
import gc
import platform
from pathlib import Path

import pandas as pd
import psutil

In [2]:
# Project paths (notebook lives in notebooks/; data is one level up)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "dataverse dataset"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Italy country code in this dataset (Milan traffic)
COUNTRY_ITALY = 39

daily_files = sorted(RAW_DIR.glob("sms-call-internet-mi-*.txt"))
print(f"Found {len(daily_files)} daily files in:\n  {RAW_DIR}")

Found 62 daily files in:
  c:\Users\HP ENVY\OneDrive\Desktop\Time Series Forecasting\dataverse dataset


## V. Hardware and software setup

Record the environment used for loading and optimization (paste into the report).

In [3]:
# Collect hardware / software details for the report
vm = psutil.virtual_memory()
setup = {
    "OS": platform.platform(),
    "Python": platform.python_version(),
    "pandas": pd.__version__,
    "CPU cores (logical)": psutil.cpu_count(logical=True),
    "RAM total (GB)": round(vm.total / 1e9, 2),
    "RAM available at start (GB)": round(vm.available / 1e9, 2),
}
for key, value in setup.items():
    print(f"{key}: {value}")

OS: Windows-11-10.0.26100-SP0
Python: 3.14.0
pandas: 3.0.3
CPU cores (logical): 4
RAM total (GB): 17.07
RAM available at start (GB): 6.03


## Inspect raw file format

Each line is tab-separated. We only need columns 0 (square), 1 (timestamp ms), and 7 (internet). Column 2 is country code (used to keep Italian records).

In [4]:
# Peek at the first file (small sample — fast)
sample_path = daily_files[0]
peek = pd.read_csv(sample_path, sep="\t", header=None, nrows=5)
peek.columns = ["square_id", "time_ms", "country", "c4", "c5", "c6", "c7", "internet"][: peek.shape[1]]
display(peek)

,square_id,time_ms,country,c4,c5,c6,c7,internet
0,1,1383260400000,0,0.081363,NaN,NaN,NaN,NaN
1,1,1383260400000,39,0.141864,0.156787,0.160938,0.052275,11.028366
2,1,1383261000000,0,0.136588,NaN,NaN,0.027300,NaN
3,1,1383261000000,33,NaN,NaN,NaN,NaN,0.026137
4,1,1383261000000,39,0.278452,0.119926,0.188777,0.133637,11.100963


In [5]:
# Helper: DataFrame memory in MB (deep=True counts actual string/object storage)
def memory_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024 ** 2)


def report_memory(label: str, df: pd.DataFrame) -> float:
    mb = memory_mb(df)
    print(f"{label}: {mb:.2f} MB  |  rows={len(df):,}  |  cols={df.shape[1]}")
    return mb

## III. Memory comparison on one daily file

**Before:** load all columns with default pandas dtypes (simulates naive handling).

**After:** read only required columns, filter Italy, downcast numeric types, convert time to `datetime64`.

In [6]:
# BEFORE optimization — full width, default dtypes (one day only)
gc.collect()
df_before = pd.read_csv(sample_path, sep="\t", header=None)
mb_before = report_memory("Before (1 day, all columns)", df_before)

Before (1 day, all columns): 295.57 MB  |  rows=4,842,625  |  cols=8


In [8]:
# AFTER optimization — same file, assignment-relevant fields only
raw_slim = pd.read_csv(
    sample_path,
    sep="\t",
    header=None,
    usecols=[0, 1, 2, 7],
    names=["square_id", "time_ms", "country", "internet"],
)

# Keep Italian mobile-internet rows; drop unused country column
df_after = raw_slim.loc[raw_slim["country"] == COUNTRY_ITALY, ["square_id", "time_ms", "internet"]].copy()
del raw_slim

# Smaller dtypes: 10k squares fit in uint16; traffic fits float32
df_after["square_id"] = df_after["square_id"].astype("uint16")
df_after["internet"] = df_after["internet"].astype("float32")
df_after["time"] = pd.to_datetime(df_after["time_ms"], unit="ms")
df_after.drop(columns="time_ms", inplace=True)

gc.collect()
mb_after = report_memory("After (1 day, optimized)", df_after)

After (1 day, optimized): 30.21 MB  |  rows=1,439,981  |  cols=3


In [9]:
# Side-by-side reduction on one file (evidence for the report)
reduction_pct = (1 - mb_after / mb_before) * 100
summary_one_day = pd.DataFrame(
    {
        "Stage": ["Before optimization", "After optimization"],
        "Memory (MB)": [round(mb_before, 2), round(mb_after, 2)],
        "Rows": [len(df_before), len(df_after)],
    }
)
display(summary_one_day)
print(f"Memory reduction on one daily file: {reduction_pct:.1f}%")

del df_before  # free RAM before processing all files
gc.collect()

,Stage,Memory (MB),Rows
0,Before optimization,295.57,4842625
1,After optimization,30.21,1439981


Memory reduction on one daily file: 89.8%


0

## I. Loading strategy (full dataset)

1. **One file at a time** — never hold all raw text in memory at once.  
2. **`usecols`** — skip SMS/call fields not needed for this assignment.  
3. **Filter `country == 39`** — Milan/Italy internet traffic only.  
4. **Downcast** — `uint16` square id, `float32` traffic, `datetime64` time.  
5. **Append** optimized daily chunks, then **Parquet** cache for Tasks 2–3.

## II. Justification of transformations

| Choice | Why it is appropriate |
|--------|------------------------|
| Drop SMS/call columns | Not used in Tasks 1–3; removes ~75% of columns. |
| Filter country 39 | Dataset encodes multiple countries per square/time; homework targets Milan. |
| `float32` for traffic | CDR counts are approximate; precision loss is negligible for EDA/forecasting. |
| `uint16` for square id | Grid has 10,000 cells (ids 1–10000). |
| Parquet output | Columnar, compressed, faster reload than 62 text files. |

In [10]:
def load_day_optimized(path: Path) -> pd.DataFrame:
    """Load one daily file with memory-friendly dtypes (used in the loop below)."""
    chunk = pd.read_csv(
        path,
        sep="\t",
        header=None,
        usecols=[0, 1, 2, 7],
        names=["square_id", "time_ms", "country", "internet"],
    )
    chunk = chunk.loc[chunk["country"] == COUNTRY_ITALY, ["square_id", "time_ms", "internet"]]
    chunk["square_id"] = chunk["square_id"].astype("uint16")
    chunk["internet"] = chunk["internet"].astype("float32")
    chunk["time"] = pd.to_datetime(chunk["time_ms"], unit="ms")
    return chunk.drop(columns="time_ms")

In [11]:
# Process every daily file sequentially (limits peak RAM)
optimized_parts = []
for i, path in enumerate(daily_files, start=1):
    day_df = load_day_optimized(path)
    optimized_parts.append(day_df)
    if i % 10 == 0 or i == len(daily_files):
        print(f"  loaded {i}/{len(daily_files)}: {path.name}  ({len(day_df):,} rows)")

traffic = pd.concat(optimized_parts, ignore_index=True)
del optimized_parts
gc.collect()

  loaded 10/62: sms-call-internet-mi-2013-11-10.txt  (1,439,982 rows)
  loaded 20/62: sms-call-internet-mi-2013-11-20.txt  (1,439,962 rows)
  loaded 30/62: sms-call-internet-mi-2013-11-30.txt  (1,439,524 rows)
  loaded 40/62: sms-call-internet-mi-2013-12-10.txt  (1,439,122 rows)
  loaded 50/62: sms-call-internet-mi-2013-12-20.txt  (1,439,119 rows)
  loaded 60/62: sms-call-internet-mi-2013-12-30.txt  (1,438,690 rows)
  loaded 62/62: sms-call-internet-mi-2014-01-01.txt  (1,438,569 rows)


0

In [12]:
# Full optimized dataset — memory and basic checks
mb_full = report_memory("After optimization (full dataset)", traffic)

print("\nDate range:", traffic["time"].min(), "→", traffic["time"].max())
print("Unique squares:", traffic["square_id"].nunique())
print("\nSample:")
display(traffic.head())

After optimization (full dataset): 1191.52 MB  |  rows=89,243,075  |  cols=3

Date range: 2013-10-31 23:00:00 → 2014-01-01 22:50:00
Unique squares: 10000

Sample:


,square_id,internet,time
0,1,11.028366,2013-10-31 23:00:00
1,1,11.100964,2013-10-31 23:10:00
2,1,10.892771,2013-10-31 23:20:00
3,1,8.622424,2013-10-31 23:30:00
4,1,8.009928,2013-10-31 23:40:00


In [13]:
# Estimate naive full-dataset memory (extrapolate from one-day baseline)
n_days = len(daily_files)
mb_naive_estimated = mb_before * n_days

summary_full = pd.DataFrame(
    {
        "Stage": [
            "Before (estimated, all days × all columns)",
            "After (measured, full optimized dataset)",
        ],
        "Memory (MB)": [round(mb_naive_estimated, 2), round(mb_full, 2)],
        "Notes": [
            f"{mb_before:.1f} MB/day × {n_days} files",
            "concatenated optimized chunks",
        ],
    }
)
display(summary_full)
print(f"Estimated reduction vs naive full load: {(1 - mb_full / mb_naive_estimated) * 100:.1f}%")

,Stage,Memory (MB),Notes
0,"Before (estimated, all days × all columns)",18325.37,295.6 MB/day × 62 files
1,"After (measured, full optimized dataset)",1191.52,concatenated optimized chunks


Estimated reduction vs naive full load: 93.5%


In [14]:
# Save compressed Parquet for fast reuse in Tasks 2 and 3
parquet_path = PROCESSED_DIR / "milan_internet_traffic.parquet"
traffic.to_parquet(parquet_path, index=False, compression="snappy")

disk_mb = parquet_path.stat().st_size / (1024 ** 2)
print(f"Saved: {parquet_path}")
print(f"On-disk size: {disk_mb:.2f} MB (Snappy compression)")

Saved: c:\Users\HP ENVY\OneDrive\Desktop\Time Series Forecasting\data\processed\milan_internet_traffic.parquet
On-disk size: 390.82 MB (Snappy compression)


In [15]:
# Fast reload path — avoids re-parsing 62 text files in later tasks
reloaded = pd.read_parquet(parquet_path)
report_memory("Reloaded from Parquet", reloaded)

Reloaded from Parquet: 1191.52 MB  |  rows=89,243,075  |  cols=3


np.float64(1191.5237255096436)

## IV. Challenges and how they were addressed

- **Large raw size (~19 GB on disk, 62 files):** Process **one day per iteration** instead of `read_csv` on all files at once.  
- **Extra fields (SMS, calls, multiple countries):** Use **`usecols`** and **filter country 39** so only homework-relevant rows/columns remain.  
- **Default `float64` / `int64` overhead:** **Downcast** to `float32` / `uint16` after filtering.  
- **Repeated slow text parsing:** Write **`milan_internet_traffic.parquet`** once; reload in later tasks.  
- **RAM limits on typical laptops:** Peak memory is roughly **one day of slim data + final concat** (~2–3 GB optimized vs tens of GB naive) — if RAM is tight, process in batches and append to Parquet with `pyarrow` dataset API (optional extension).

---

**Report checklist (Task 1):** Copy tables printed above, environment block, strategy bullets, and justification table into your PDF. Cite Dataverse [2] and Barlacchi et al. [1].